# 04 — Temporal tuning, three models, A/B ablation and freeze
Never open 2024 descriptive results before this notebook freezes the model system. The predeclared natural-prevalence samples preserve threshold-copy grouping.

The interrupted run left partial artifacts under the previous `runs/20261011-full-01/` prefix; keep them unchanged. This run uses `RUN_ID=20261012-full-01` and `MODEL_ATTEMPT=attempt-01`, with an independent empty output prefix. If interrupted, preserve this attempt and select the next unused attempt ID in both Notebooks 04 and 05 before retrying. Do not delete or overwrite earlier model artifacts.

In [ ]:
from pathlib import Path
import os
import sys

from IPython.display import display
from pyspark.sql import SparkSession

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path[:0] = [str(PROJECT_ROOT / "src"), str(PROJECT_ROOT)]

from rainfall.config import load_config
from rainfall.runtime import spark_action

BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
RUN_ID = os.environ.get("RUN_ID", "20261012-full-01")
MODEL_ATTEMPT = os.environ.get("MODEL_ATTEMPT", "attempt-01")
GIT_COMMIT = os.environ.get("GIT_COMMIT")
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
config = load_config(PROJECT_ROOT / "config" / "experiment.json")
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.ansi.enabled", "false")

In [ ]:
from rainfall.experiments import fit_models
from rainfall.features import feature_dictionary

display(feature_dictionary(config))
from scripts.run_quality_audit import read_manifest
manifest_path = PROJECT_ROOT / "manifests" / "input_manifest.json"
manifest, manifest_sha256 = read_manifest(manifest_path)
MODEL_ROOT = f"{ROOT}/models/{MODEL_ATTEMPT}"
model_path = spark._jvm.org.apache.hadoop.fs.Path(MODEL_ROOT)
model_fs = model_path.getFileSystem(spark._jsc.hadoopConfiguration())
if model_fs.exists(model_path):
    raise FileExistsError(
        f"{MODEL_ROOT} already exists. Preserve it and use a new MODEL_ATTEMPT "
        "in both Notebooks 04 and 05 after reviewing the prior attempt."
    )
with spark_action(spark, RUN_ID, "model_selection_calibration_freeze"):
    base = spark.read.parquet(f"{ROOT}/base_features")
    frozen = fit_models(
        base, config, MODEL_ROOT,
        manifest_sha256=manifest_sha256,
        git_commit=GIT_COMMIT,
    )
    displayed = [
        "primary_thresholds", "pilot_expanded_n", "final_train_expanded_n",
        "training_scope", "selected", "use_calibration", "selection_ece",
        "probability_cutoff", "probability_cutoff_policy", "config_sha256",
    ]
    display({key: frozen[key] for key in displayed})
    print(f"Model attempt: {MODEL_ATTEMPT}")
    print(f"Frozen model manifest: {MODEL_ROOT}/frozen_manifest")

In [ ]:
spark.stop()